In [13]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

device = "cuda" if torch.cuda.is_available() else "cpu" # Chuẩn bị đồ nghề và xem máy chạy bằng GPU hay CPU.
print("Using device:", device)

# Có GPU → dùng cuda - Không có GPU → dùng cpu

Using device: cpu


In [14]:
CANDIDATE_INPUT = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/08_candidate_texts.xlsx"
JOB_INPUT = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/09_job_texts.xlsx"

CANDIDATE_OUTPUT = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/10_candidate_embeddings.parquet"
JOB_OUTPUT = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/11_job_embeddings.parquet"

MODEL_NAME = "intfloat/multilingual-e5-base" # Đây là model dùng để biến text thành vector.
BATCH_SIZE = 16 # mỗi lần xử lý 16 đoạn text.
MAX_LENGTH = 512 # mỗi text tối đa 512 token: Mỗi 1 text có thể dài nhiều chữ, nhiều câu, tối đa bị cắt ở 512.

Định nghĩa hàm tạo embedding
Model cho mỗi từ một vector.
mean_pooling gom các vector từ lại thành một vector chung cho cả câu/CV/job. (TOKEN CÓ THỂ HIỂU THEO NGHĨA LÀ TỪNG TỪ)

Token = mảnh chữ model cắt ra
Token vector = dãy số biểu diễn mảnh chữ đó
Text vector = trung bình các token vector

2. attention_mask là cái gì?

Khi xử lý batch, các câu có độ dài khác nhau.

Ví dụ:

Text 1: "java mysql"
Text 2: "frontend developer react typescript css"

Để đưa vào model cùng lúc, tokenizer phải padding, tức là đệm thêm token rỗng cho câu ngắn.

Ví dụ:

- Text 1: java mysql [PAD] [PAD] [PAD]
- Text 2: frontend developer react typescript css

attention_mask cho biết token nào là thật, token nào là padding.

- java      → 1
- mysql     → 1
- [PAD]     → 0
- [PAD]     → 0

Trong mean_pooling, mask dùng để:

Chỉ tính trung bình token thật, bỏ qua token padding.

Nếu không bỏ padding, vector sẽ bị nhiễu.

4. l2_normalize làm gì?
def l2_normalize(x):
    return torch.nn.functional.normalize(x, p=2, dim=1)

Nó đưa vector về cùng độ dài chuẩn.

Tưởng tượng có 2 vector:

- A dài 100
- B dài 2

Nếu so trực tiếp thì vector dài có thể lấn át.

Normalize làm cho vector có cùng “độ dài”, để khi so sánh bằng cosine similarity thì công bằng hơn.

Nói mầm non:

đưa tất cả vector về cùng một thước đo

Ví dụ đời thường

CV text:

backend developer with java mysql

Model không hiểu nguyên câu một phát. Nó tách ra:

backend | developer | java | mysql

Mỗi từ có một vector riêng.

mean_pooling nói:

cho tao lấy trung bình cả đám này
để ra 1 vector đại diện cho toàn CV

l2_normalize nói:

đưa vector này về thang đo chuẩn
để lát nữa so với job cho công bằng

encode_texts nói:

tao làm quy trình đó cho toàn bộ candidate và job

Với A:

A = [3, 4]
độ dài A = sqrt(3² + 4²) = 5

A_normalized = [3/5, 4/5] = [0.6, 0.8]

Với B:

B = [30, 40]
độ dài B = sqrt(30² + 40²) = 50

B_normalized = [30/50, 40/50] = [0.6, 0.8]

Sau normalize:

A_normalized = [0.6, 0.8]
B_normalized = [0.6, 0.8]

Tức là dù B ban đầu to hơn, sau khi chuẩn hóa thì cả hai về cùng thang đo.

In [15]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    sum_embeddings = masked_embeddings.sum(dim=1)
    sum_mask = mask.sum(dim=1).clamp(min=1e-9)
    return sum_embeddings / sum_mask

# mỗi từ có một điểm số riêng
# → cộng điểm các từ lại
# → chia trung bình
# → ra điểm đại diện cho cả câu

def l2_normalize(x): # Đưa vector về cùng thang đo, để so sánh công bằng hơn.
    return torch.nn.functional.normalize(x, p=2, dim=1)

# hàm chính
def encode_texts(texts, tokenizer, model, batch_size=16, max_length=512):
# 1. chia text thành batch nhỏ
# 2. tokenize text
# 3. đưa token vào model
# 4. model trả vector từng token
# 5. mean_pooling gom thành vector cả text
# 6. normalize vector
# 7. lưu tất cả vector lại

    all_embeddings = [] # nơi chứa kết quả
    model.eval() # đưa model về chế độ dự đoán, không huấn luyện.

    #Tạo list để chứa vector. model.eval() nghĩa là đưa model về chế độ dự đoán, không huấn luyện.

    with torch.no_grad(): # không cần tính toán gradient, tiết kiệm tài nguyên.

        for start in range(0, len(texts), batch_size): # chia text thành batch nhỏ
            batch_texts = texts[start:start + batch_size]
            batch_texts = [str(t) if pd.notna(t) else "" for t in batch_texts] # Nếu text bị rỗng hoặc NaN, nó đổi thành chuỗi rỗng "". Tránh lỗi khi đưa vào tokenizer.

            # biến text thành dữ liệu model hiểu được.
            inputs = tokenizer(
                batch_texts,
                padding=True, # điền thêm token để tất cả text trong batch có cùng độ dài, giúp xử lý hiệu quả hơn. câu ngắn được đệm thêm cho bằng nhau.
                truncation=True, # cắt text nếu nó dài hơn max_length, tránh lỗi khi đưa vào model: câu quá dài thì cắt bớt.
                max_length=max_length,
                return_tensors="pt" # trả về dưới dạng PyTorch tensors, phù hợp với model.
            ).to(device) # đưa dữ liệu vào GPU hoặc CPU tùy máy.

            # TÓM LẠI LÀ: biến text thành dạng số mà model hiểu được, chuẩn bị cho bước tiếp theo.

            outputs = model(**inputs) # đưa token vào model, model đọc và trả về vector cho từng token trong text.

            # mean_pooling lấy trung bình vector của các token thật. mỗi text → 1 vector
            embeddings = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"]) 

            # Đưa vector về cùng thang đo để lát nữa tính cosine similarity cho công bằng.
            embeddings = l2_normalize(embeddings)

            # lưu vector của batch vào list kết quả, chuyển về CPU và numpy để dễ lưu trữ sau này.
            all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

In [16]:
candidate_df = pd.read_excel(CANDIDATE_INPUT)
job_df = pd.read_excel(JOB_INPUT)

print("Candidate columns:", candidate_df.columns.tolist())
print("Job columns:", job_df.columns.tolist())

Candidate columns: ['candidate_id', 'candidate_text']
Job columns: ['job_id', 'job_text']


In [17]:
candidate_cols = {c.lower(): c for c in candidate_df.columns}
job_cols = {c.lower(): c for c in job_df.columns}

cand_id_col = candidate_cols.get("candidate_id", candidate_cols.get("id"))
cand_text_col = candidate_cols.get("candidate_text", candidate_cols.get("text"))

job_id_col = job_cols.get("job_id", job_cols.get("id"))
job_text_col = job_cols.get("job_text", job_cols.get("text"))

if cand_id_col is None or cand_text_col is None:
    raise ValueError(
        f"Không tìm thấy cột candidate_id/candidate_text trong {CANDIDATE_INPUT}. "
        f"Columns hiện có: {candidate_df.columns.tolist()}"
    )

if job_id_col is None or job_text_col is None:
    raise ValueError(
        f"Không tìm thấy cột job_id/job_text trong {JOB_INPUT}. "
        f"Columns hiện có: {job_df.columns.tolist()}"
    )

candidate_df = candidate_df[[cand_id_col, cand_text_col]].copy()
candidate_df.columns = ["candidate_id", "candidate_text"]
candidate_df["candidate_text"] = candidate_df["candidate_text"].fillna("").astype(str).str.strip()

job_df = job_df[[job_id_col, job_text_col]].copy()
job_df.columns = ["job_id", "job_text"]
job_df["job_text"] = job_df["job_text"].fillna("").astype(str).str.strip()

print("Candidate rows:", len(candidate_df))
print("Job rows:", len(job_df))

Candidate rows: 20
Job rows: 80


Load model:

intfloat/multilingual-e5-base

Gồm:

tokenizer: tách text thành token cho model đọc.
model: tạo embedding vector.

Nói mầm non:

Gọi con AI embedding ra để chuẩn bị biến text thành vector.

In [18]:
print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)

Loading model: intfloat/multilingual-e5-base


Lấy toàn bộ candidate_text, đưa vào hàm encode_texts.

Output:

candidate_embeddings

là ma trận vector.

Ví dụ có 3 candidate, vector dimension 768:

shape = (3, 768)

In [19]:
# Nếu dùng cùng một model embedding cho cùng kiểu xử lý, thì nó luôn trả về vector cùng số chiều.

print("Encoding candidate_text...")
candidate_embeddings = encode_texts(
    candidate_df["candidate_text"].tolist(),
    tokenizer=tokenizer,
    model=model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

candidate_embeddings.shape

Encoding candidate_text...


(20, 768)

Tương tự cell 6, nhưng làm cho job.

Output:

job_embeddings

Ví dụ có 10 job:

shape = (10, 768)

Nói mầm non:

Biến text của job thành vector.


In [20]:
print("Encoding job_text...")
job_embeddings = encode_texts(
    job_df["job_text"].tolist(),
    tokenizer=tokenizer,
    model=model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

job_embeddings.shape

Encoding job_text...


(80, 768)

Nó tạo bảng output candidate gồm:

candidate_id
candidate_text
embedding_model
embedding_dim
embedding_vector

Tương tự cho job.

Nói mầm non:

Lấy vector vừa tạo, nhét lại vào bảng candidate/job.

In [21]:
candidate_out = candidate_df.copy()
candidate_out["embedding_model"] = MODEL_NAME
candidate_out["embedding_dim"] = candidate_embeddings.shape[1]
candidate_out["embedding_vector"] = [vec.tolist() for vec in candidate_embeddings]

job_out = job_df.copy()
job_out["embedding_model"] = MODEL_NAME
job_out["embedding_dim"] = job_embeddings.shape[1]
job_out["embedding_vector"] = [vec.tolist() for vec in job_embeddings]

candidate_out.head(2), job_out.head(2)

(  candidate_id                                     candidate_text  \
 0         C001  query: Candidate group: Software Development. ...   
 1         C002  query: Candidate group: Software Development. ...   
 
                  embedding_model  embedding_dim  \
 0  intfloat/multilingual-e5-base            768   
 1  intfloat/multilingual-e5-base            768   
 
                                     embedding_vector  
 0  [0.004771376959979534, 0.05155104398727417, -0...  
 1  [0.01015084981918335, 0.04290420189499855, -0....  ,
   job_id                                           job_text  \
 0   J001  passage: Job title: Angular Frontend Engineer....   
 1   J002  passage: Job title: Next.js Web Developer. Job...   
 
                  embedding_model  embedding_dim  \
 0  intfloat/multilingual-e5-base            768   
 1  intfloat/multilingual-e5-base            768   
 
                                     embedding_vector  
 0  [-0.00839162990450859, 0.06643076241016388, 0....

In [22]:
candidate_out.to_parquet(CANDIDATE_OUTPUT, index=False)
job_out.to_parquet(JOB_OUTPUT, index=False)

print(f"Saved -> {CANDIDATE_OUTPUT}")
print(f"Saved -> {JOB_OUTPUT}")

Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/10_candidate_embeddings.parquet
Saved -> /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Embedding/data_outputs/11_job_embeddings.parquet


In [23]:
cand_check = pd.read_parquet(CANDIDATE_OUTPUT)
job_check = pd.read_parquet(JOB_OUTPUT)

print(cand_check.head(2))
print(job_check.head(2))

print(cand_check["embedding_dim"].unique())
print(job_check["embedding_dim"].unique())

print(len(cand_check.loc[0, "embedding_vector"]))
print(len(job_check.loc[0, "embedding_vector"]))

  candidate_id                                     candidate_text  \
0         C001  query: Candidate group: Software Development. ...   
1         C002  query: Candidate group: Software Development. ...   

                 embedding_model  embedding_dim  \
0  intfloat/multilingual-e5-base            768   
1  intfloat/multilingual-e5-base            768   

                                    embedding_vector  
0  [0.004771376959979534, 0.05155104398727417, -0...  
1  [0.01015084981918335, 0.04290420189499855, -0....  
  job_id                                           job_text  \
0   J001  passage: Job title: Angular Frontend Engineer....   
1   J002  passage: Job title: Next.js Web Developer. Job...   

                 embedding_model  embedding_dim  \
0  intfloat/multilingual-e5-base            768   
1  intfloat/multilingual-e5-base            768   

                                    embedding_vector  
0  [-0.00839162990450859, 0.06643076241016388, 0....  
1  [-0.008345445618